# NBA 3.1 (v6) — Final HCP Universe Creation

Builds the NBA 3.1 HCP universe as defined in `NBA_3.1_HCP_Universe_Presentation_V6_ZS_Recommended.md` (Section 1.2) and writes it as a parquet file `HCP_Universe` to `/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume`.

| Criterion | Definition | Source |
|:--|:--|:--|
| **C1** Core prescribers | Set 1 (T2D x SGLT2/GLP1) ∪ Set 3 (CKD x SGLT2/GLP1), cumulative-NBRX decile >= 2, 24-month lookback | `pace_ker_in_house_schema.lrx_nbrx` |
| **C2** Kerendia writers | NBRX > 0, all claim types (PD+RV+RJ), full history up to the lookback end date | Snowflake `STG_CAR_IQV_LAAD_FCT_RX` x `DIM_PRD` |
| **C3** Specialists | `SPECIALTY_GROUP` in (NEPH, CARD) within the lookback | `lrx_nbrx` |
| **C4** Targeting / BPT | `CP_KER_HCP`, FF in CR1/CR2/CR4/CR5, Product = KERENDIA | Snowflake `PHCDW_TARGETING.CP_KER_HCP` |
| **C5** Call history (Section 4.1) | Submitted Kerendia calls since `CALL_START_DATE`; incremental HCPs added when `INCLUDE_C5_IN_UNIVERSE = Yes` (default) | Snowflake `CPH_DB_PROD.MODEL_V2.FACT_CALL` x `MAP_CUSTOMER_XREF` |

**Dynamic dates:** `LOOKBACK_END` = last complete calendar month in `lrx_nbrx`; `LOOKBACK_START` = `LOOKBACK_END` minus `LOOKBACK_MONTHS` (default 24) + 1 day. Both can be overridden with widgets.

**Latest result (run 2026-10-27, window 2024-10-01 – 2026-09-30, INCLUDE_C5_IN_UNIVERSE = Yes):**

| Step | HCPs in criterion | Universe total | Incremental |
|:--|--:|--:|--:|
| C1 Core (D2+) | 242,978 | 242,978 | - |
| + C2 Kerendia writers | 62,241 | 255,289 | +12,311 |
| + C3 NEPH (15,063) / CARD (51,737) | 66,800 | 291,829 | +36,540 |
| + C4 Targeting / BPT | 78,813 | 294,704 | +2,875 |
| + C5 Called since 2023 | 147,571 | **314,920** | +20,216 |

Previous reference (md v6, C1-C4 only): C1 242,868 | C2 61,583 | C3 66,714 | C4 78,798 | Total 294,244. The source tables are refreshed every week, so counts move a little with each run.

In [0]:
%run "/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
# ============================================================
# Parameters & dynamic date window
# ============================================================
import builtins
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F, Window

# --- Widgets (leave dates blank to auto-derive from the latest data) ---
dbutils.widgets.text("LOOKBACK_END", "", "Lookback end (YYYY-MM-DD, blank = auto)")
dbutils.widgets.text("LOOKBACK_MONTHS", "24", "Lookback months")
dbutils.widgets.text("DECILE_CUTOFF", "2", "C1 minimum decile")
dbutils.widgets.text("CALL_START_DATE", "2023-01-01", "C5 call start date")
dbutils.widgets.dropdown("INCLUDE_C5_IN_UNIVERSE", "Yes", ["No", "Yes"], "Add C5 called HCPs to universe")

CATALOG      = "ph-com-ai-us-prod-catalog-2553213839599380"
SOURCE_TABLE = f"`{CATALOG}`.pace_ker_in_house_schema.lrx_nbrx"
MAPPING_TABLE = f"`{CATALOG}`.kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING"
OUTPUT_DIR   = f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_1_volume"
OUTPUT_PATH  = f"{OUTPUT_DIR}/HCP_Universe"

INDICATION_COL = "INDICATION_GROUP_APP1"
PRODUCT_COL    = "PRODUCT_CLASS"
METRIC_COL     = "NBRX"
CORE_SETS = {                      # C1 core = Set 1 ∪ Set 3
    "Set1_T2D_SGLT2_GLP1": ("T2D", ["SGLT2", "GLP1"]),
    "Set3_CKD_SGLT2_GLP1": ("CKD", ["SGLT2", "GLP1"]),
}
C3_SPECIALTIES = ["NEPH", "CARD"]

LOOKBACK_MONTHS = int(dbutils.widgets.get("LOOKBACK_MONTHS"))
DECILE_CUTOFF   = int(dbutils.widgets.get("DECILE_CUTOFF"))
CALL_START_DATE = dbutils.widgets.get("CALL_START_DATE").strip()
INCLUDE_C5      = dbutils.widgets.get("INCLUDE_C5_IN_UNIVERSE") == "Yes"

# --- Dynamic end date: last COMPLETE calendar month available in lrx_nbrx ---
end_override = dbutils.widgets.get("LOOKBACK_END").strip()
if end_override:
    lookback_end = datetime.strptime(end_override, "%Y-%m-%d").date()
else:
    max_dt = spark.table(SOURCE_TABLE).agg(F.max("WEEK_ENDING_DATE")).collect()[0][0]
    month_end = (max_dt.replace(day=1) + relativedelta(months=1)) - relativedelta(days=1)
    # If the data does not reach the month end, step back to the previous complete month
    lookback_end = max_dt if max_dt == month_end else (max_dt.replace(day=1) - relativedelta(days=1))

lookback_start = (lookback_end + relativedelta(days=1)) - relativedelta(months=LOOKBACK_MONTHS)
LOOKBACK_START = lookback_start.strftime("%Y-%m-%d")
LOOKBACK_END   = lookback_end.strftime("%Y-%m-%d")

print(f"Claims lookback window : {LOOKBACK_START} to {LOOKBACK_END} ({LOOKBACK_MONTHS} months)")
print(f"C1 decile cutoff       : D{DECILE_CUTOFF}+")
print(f"C2 Kerendia history    : full history up to {LOOKBACK_END}")
print(f"C5 call window         : {CALL_START_DATE} to {LOOKBACK_END}")
print(f"Include C5 in universe : {INCLUDE_C5}")
print(f"Output                 : {OUTPUT_PATH}")

In [0]:
# ============================================================
# Load lrx_nbrx within the lookback window
# ============================================================
lrx = (
    spark.table(SOURCE_TABLE)
    .filter((F.col("WEEK_ENDING_DATE") >= LOOKBACK_START) & (F.col("WEEK_ENDING_DATE") <= LOOKBACK_END))
    .withColumn("PROVIDER_ID", F.col("PROVIDER_ID").cast("string"))
)
print(f"Unique HCPs in lookback: {lrx.select('PROVIDER_ID').distinct().count():,}")

In [0]:
# ============================================================
# C1: cumulative-volume NBRX deciles (each decile ~10% of NBRX)
# D1 = lowest volume, D10 = highest. Tiebreaker: PROVIDER_ID asc.
# ============================================================
def compute_deciles(df, indication, pool_products):
    hcp_vol = (
        df.filter(F.col(INDICATION_COL).contains(indication) & F.col(PRODUCT_COL).isin(pool_products))
        .groupBy("PROVIDER_ID").agg(F.sum(METRIC_COL).alias("TOTAL_NBRX"))
    )
    total_vol = hcp_vol.agg(F.sum("TOTAL_NBRX")).collect()[0][0]
    if not total_vol:
        return None
    w = Window.orderBy(F.asc("TOTAL_NBRX"), F.asc("PROVIDER_ID"))
    return (
        hcp_vol.withColumn("CUM_NBRX", F.sum("TOTAL_NBRX").over(w))
        .withColumn("DECILE", F.least((F.col("CUM_NBRX") / F.lit(float(total_vol) / 10.0)).cast("int") + 1, F.lit(10)))
        .drop("CUM_NBRX")
    )

core_parts = []
for set_name, (ind, products) in CORE_SETS.items():
    dec = compute_deciles(lrx, ind, products)
    if dec is None:
        raise ValueError(f"{set_name}: no NBRX volume in the lookback window")
    part = dec.filter(F.col("DECILE") >= DECILE_CUTOFF).select("PROVIDER_ID")
    print(f"{set_name}: {dec.count():,} HCPs total | D{DECILE_CUTOFF}+: {part.count():,}")
    core_parts.append(part)

c1 = core_parts[0].unionByName(core_parts[1]).distinct().cache()
c1_ct = c1.count()
print(f"\nC1 Core (Set1 ∪ Set3, D{DECILE_CUTOFF}+): {c1_ct:,}")

In [0]:
# ============================================================
# C2: Kerendia writers from raw STG_CAR_IQV_LAAD_FCT_RX
# - All claim types (PD, RV, RJ), full history up to LOOKBACK_END
# - NBRX: first-ever claim per patient-product, or gap > 365 days
#   (same logic as Kerendia_NBRX_From_Raw_FCT_RX)
# ============================================================
query_c2 = f"""
    SELECT DISTINCT
        A.CLAIM_ID,
        A.PATIENT_ID,
        TO_DATE(A.SVC_DT) AS SVC_DT,
        A.PROVIDER_ID,
        A.CLAIM_TYPE,
        TRIM(UPPER(B.PRODUCT_GROUP)) AS PRODUCT_GROUP
    FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B USING (NDC_CD)
    WHERE TRIM(UPPER(B.PRODUCT_GROUP)) = 'KERENDIA'
      AND TO_DATE(A.SVC_DT) <= '{LOOKBACK_END}'
"""
ker_claims = Get_Data_Snowflakes(query_c2).withColumn("PROVIDER_ID", F.col("PROVIDER_ID").cast("string"))

w_day = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP", "SVC_DT").orderBy("CLAIM_ID")
w_pat = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP").orderBy("SVC_DT")
ker_claims = (
    ker_claims
    .withColumn("CLAIM_RANK", F.row_number().over(w_day))
    .withColumn("MIN_SVC_DT", F.min("SVC_DT").over(w_pat))
    .withColumn("LAG_SVC_DT", F.lag("SVC_DT").over(w_pat))
    .withColumn(
        "NBRX_FLAG",
        F.when((F.col("MIN_SVC_DT") == F.col("SVC_DT")) & (F.col("CLAIM_RANK") == 1), 1)
         .when(F.col("LAG_SVC_DT").isNotNull() & (F.col("LAG_SVC_DT") < F.expr("date_add(SVC_DT, -365)")) & (F.col("CLAIM_RANK") == 1), 1)
         .otherwise(0)
    )
)

ker_hcp = ker_claims.groupBy("PROVIDER_ID").agg(
    F.sum("NBRX_FLAG").alias("NBRX_KERENDIA_OVERALL"),
    F.min("SVC_DT").alias("FIRST_KERENDIA_CLAIM"),
    F.max("SVC_DT").alias("LAST_KERENDIA_CLAIM"),
)
c2 = ker_hcp.filter(F.col("NBRX_KERENDIA_OVERALL") > 0).select("PROVIDER_ID").distinct().cache()
c2_ct = c2.count()
rng = ker_hcp.agg(F.min("FIRST_KERENDIA_CLAIM"), F.max("LAST_KERENDIA_CLAIM")).collect()[0]
print(f"Kerendia claim history: {rng[0]} to {rng[1]}")
print(f"C2 Kerendia writers (NBRX > 0, all claim types): {c2_ct:,}")

In [0]:
# ============================================================
# C3: Nephrologists + Cardiologists seen in lrx_nbrx (lookback)
# ============================================================
spec = lrx.select("PROVIDER_ID", "SPECIALTY_GROUP").distinct()
for s in C3_SPECIALTIES:
    print(f"C3 {s}: {spec.filter(F.col('SPECIALTY_GROUP') == s).select('PROVIDER_ID').distinct().count():,}")

c3 = spec.filter(F.col("SPECIALTY_GROUP").isin(C3_SPECIALTIES)).select("PROVIDER_ID").distinct().cache()
c3_ct = c3.count()
print(f"C3 Specialists (NEPH ∪ CARD): {c3_ct:,}")

In [0]:
# ============================================================
# C4: latest Targeting / BPT list from Snowflake CP_KER_HCP
# BH_ID -> PROVIDER_ID via SOURCE_PROVIDER_BHID_NPI_MAPPING
# ============================================================
query_c4 = """
    SELECT DISTINCT BAYER_ID AS BH_ID
    FROM PHCDW.PHCDW_TARGETING.CP_KER_HCP
    WHERE FF IN ('CR1','CR2','CR4','CR5')
      AND (UPPER(PRODUCT_1) = 'KERENDIA' OR UPPER(PRODUCT_2) = 'KERENDIA')
"""
id_map = spark.table(MAPPING_TABLE).select(
    F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"),
    F.col("BH_ID").cast("string").alias("BH_ID"),
).distinct()

c4_bhids = Get_Data_Snowflakes(query_c4).select(F.col("BH_ID").cast("string").alias("BH_ID")).distinct()
c4 = c4_bhids.join(id_map, on="BH_ID", how="inner").select("PROVIDER_ID").distinct().cache()
c4_ct = c4.count()
print(f"C4 BH_IDs from CP_KER_HCP: {c4_bhids.count():,}")
print(f"C4 mapped PROVIDER_IDs:    {c4_ct:,}")

In [0]:
# ============================================================
# Base universe = C1 ∪ C2 ∪ C3 ∪ C4, with incremental buildup
# ============================================================
print(f"{'Step':<28} {'Total':>10} {'Incremental':>12}")
print("-" * 52)
running, running_ct = c1, c1_ct
print(f"{'C1 Core':<28} {running_ct:>10,} {'-':>12}")
for label, df in [("+ C2 Kerendia writers", c2), ("+ C3 NEPH/CARD", c3), ("+ C4 Targeting/BPT", c4)]:
    prev = running_ct
    running = running.unionByName(df).distinct()
    running_ct = running.count()
    print(f"{label:<28} {running_ct:>10,} {'+' + format(running_ct - prev, ','):>12}")

base_universe = running.cache()
base_ct = base_universe.count()
print(f"\nBase universe (C1-C4): {base_ct:,}  (reference Oct 2026 run: 294,244)")

## C5: Kerendia Call History (Section 4.1 method)

HCPs with **submitted Kerendia calls** (`STAT = 'Submitted_vod'`, `DETL_PROD` = KERENDIA, `BH_ID NOT LIKE '%BHO%'`) from `CALL_START_DATE` to `LOOKBACK_END`, mapped BH_ID -> PROVIDER_ID.

* `C5_FLAG` is always computed and written to the output.
* The **incremental** C5 HCPs (called but not in C1-C4) are added to the universe when the widget `INCLUDE_C5_IN_UNIVERSE = Yes` (default). Set it to `No` to keep the universe equal to C1 ∪ C2 ∪ C3 ∪ C4.

In [0]:
# ============================================================
# C5: Kerendia called HCPs (CPH_DB_PROD.MODEL_V2)
# Uses the Python Snowflake connector (credentials from 00_connection)
# ============================================================
import snowflake.connector

query_c5 = f"""
    SELECT DISTINCT B.BAYER_CUST_ID AS BH_ID
    FROM MODEL_V2.FACT_CALL A
    INNER JOIN MODEL_V2.MAP_CUSTOMER_XREF B
        ON A.SOURCE_CUST_ID = B.SOURCE_CUST_ID
    WHERE UPPER(A.STAT) = 'SUBMITTED_VOD'
      AND UPPER(A.DETL_PROD) = 'KERENDIA'
      AND B.BAYER_CUST_ID NOT LIKE '%BHO%'
      AND A.CALL_DATE >= '{CALL_START_DATE}'
      AND A.CALL_DATE <= '{LOOKBACK_END}'
"""

conn = snowflake.connector.connect(
    user=snowflake_user,
    account=snowflake_account,
    private_key=p_key,
    role="BAY_CPH_CDP_PROD_CR_US_EAST_1_MVXPF_MVXPF",
    warehouse="PROD_CYRUS_BI_WH",
    database="CPH_DB_PROD",
    schema="MODEL_V2",
)
try:
    cur = conn.cursor()
    cur.execute(query_c5)
    c5_pdf = cur.fetch_pandas_all()
finally:
    conn.close()

if len(c5_pdf) == 0:
    c5 = spark.createDataFrame([], "PROVIDER_ID string")
    c5_bh_ct = 0
else:
    c5_bh = spark.createDataFrame(c5_pdf).select(F.col("BH_ID").cast("string").alias("BH_ID")).distinct()
    c5_bh_ct = c5_bh.count()
    c5 = c5_bh.join(id_map, on="BH_ID", how="inner").select("PROVIDER_ID").distinct()
c5 = c5.cache()
c5_ct = c5.count()

c5_incremental = c5.join(base_universe, on="PROVIDER_ID", how="left_anti").cache()
c5_incr_ct = c5_incremental.count()

print(f"C5 called BH_IDs:                 {c5_bh_ct:,}")
print(f"C5 mapped PROVIDER_IDs:           {c5_ct:,}")
print(f"C5 already in C1-C4 universe:     {c5_ct - c5_incr_ct:,}")
print(f"C5 incremental (not in C1-C4):    {c5_incr_ct:,}")
print(f"Added to universe this run:       {c5_incr_ct if INCLUDE_C5 else 0:,} (INCLUDE_C5_IN_UNIVERSE = {INCLUDE_C5})")

In [0]:
# ============================================================
# Final universe with criterion flags
# ============================================================
universe_ids = base_universe.unionByName(c5_incremental).distinct() if INCLUDE_C5 else base_universe

hcp_universe = universe_ids
for flag, df in [("C1_FLAG", c1), ("C2_FLAG", c2), ("C3_FLAG", c3), ("C4_FLAG", c4), ("C5_FLAG", c5)]:
    hcp_universe = hcp_universe.join(df.withColumn(flag, F.lit(1)), on="PROVIDER_ID", how="left")

FLAGS = ["C1_FLAG", "C2_FLAG", "C3_FLAG", "C4_FLAG", "C5_FLAG"]
hcp_universe = (
    hcp_universe.fillna(0, subset=FLAGS)
    .select("PROVIDER_ID", *[F.col(f).cast("int").alias(f) for f in FLAGS])
    .cache()
)

total_ct = hcp_universe.count()
print(f"Final HCP universe: {total_ct:,}")
display(hcp_universe.agg(*[F.sum(f).alias(f) for f in FLAGS]))

In [0]:
# ============================================================
# Validation checks
# ============================================================
distinct_ct = hcp_universe.select("PROVIDER_ID").distinct().count()
null_ids = hcp_universe.filter(F.col("PROVIDER_ID").isNull()).count()
no_flag = hcp_universe.filter(builtins.sum(F.col(f) for f in ["C1_FLAG", "C2_FLAG", "C3_FLAG", "C4_FLAG"]) + (F.col("C5_FLAG") if INCLUDE_C5 else F.lit(0)) == 0).count()
c2_missing = c2.join(hcp_universe, on="PROVIDER_ID", how="left_anti").count()

assert distinct_ct == total_ct, f"Duplicate PROVIDER_IDs: {total_ct - distinct_ct:,}"
assert null_ids == 0, f"{null_ids:,} null PROVIDER_IDs"
assert no_flag == 0, f"{no_flag:,} HCPs have no qualifying flag"
assert c2_missing == 0, f"{c2_missing:,} Kerendia writers missing from the universe"

for name, df_ct in [("C1", c1_ct), ("C2", c2_ct), ("C3", c3_ct), ("C4", c4_ct)]:
    flag_ct = hcp_universe.filter(F.col(f"{name}_FLAG") == 1).count()
    assert flag_ct == df_ct, f"{name}_FLAG count {flag_ct:,} != criterion count {df_ct:,}"

print("All checks passed: unique PROVIDER_IDs, no nulls, every HCP qualifies, 100% of C2 writers captured.")
print(f"Window {LOOKBACK_START} to {LOOKBACK_END} | Total HCPs: {total_ct:,}")

In [0]:
# ============================================================
# Write parquet to the NBA 3.1 volume (overwrite with latest run)
# ============================================================
(
    hcp_universe.orderBy("PROVIDER_ID")
    .coalesce(1)
    .write.mode("overwrite")
    .parquet(OUTPUT_PATH)
)

saved = spark.read.parquet(OUTPUT_PATH)
print(f"Saved to: {OUTPUT_PATH}")
print(f"Rows read back: {saved.count():,} | Columns: {saved.columns}")
assert saved.count() == total_ct